In [11]:
import os
import json
import time
import pandas as pd
from google.cloud import bigquery
from dotenv import load_dotenv
from vertexai import generative_models as genai
import vertexai
from google.cloud import discoveryengine_v1 as discoveryengine
from vertexai import generative_models as genai  # Añadir esta línea
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    Tool,
)
import inspect
import logging
import warnings
# General
from IPython.display import HTML, Markdown, display
import plotly.graph_objects as go
# Main
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples, PointwiseMetric

In [2]:
load_dotenv()  # Carga las variables desde .env al entorno
client = bigquery.Client(project='dataton-2024-team-01-cofares')
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "europe-west4"
EXPERIMENT = "rag-eval-01"

# Inicializa el cliente de Discovery Engine
discovery_client = discoveryengine.RankServiceClient() 

vertexai.init(project=PROJECT_ID, location=LOCATION)

logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [ ]:
# ----------------------Helper functions----------------------

def print_doc(function):
    print(f"{function.__name__}:\n{inspect.getdoc(function)}\n")


def display_eval_report(eval_result, metrics=None):
    """Display the evaluation results."""

    title, summary_metrics, report_df = eval_result
    metrics_df = pd.DataFrame.from_dict(summary_metrics, orient="index").T
    if metrics:
        metrics_df = metrics_df.filter(
            [
                metric
                for metric in metrics_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )
        report_df = report_df.filter(
            [
                metric
                for metric in report_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )

    # Display the title with Markdown for emphasis
    display(Markdown(f"## {title}"))

    # Display the metrics DataFrame
    display(Markdown("### Summary Metrics"))
    display(metrics_df)

    # Display the detailed report DataFrame
    display(Markdown("### Report Metrics"))
    display(report_df)


def display_explanations(df, metrics=None, n=1):
    style = "white-space: pre-wrap; width: 800px; overflow-x: auto;"
    df = df.sample(n=n)
    if metrics:
        df = df.filter(
            ["instruction", "context", "reference", "completed_prompt", "response"]
            + [
                metric
                for metric in df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )

    for index, row in df.iterrows():
        for col in df.columns:
            display(HTML(f"{col}: {row[col]}"))
        display(HTML(""))


def plot_radar_plot(eval_results, max_score=5, metrics=None):
    fig = go.Figure()

    for eval_result in eval_results:
        title, summary_metrics, report_df = eval_result

        if metrics:
            summary_metrics = {
                k: summary_metrics[k]
                for k, v in summary_metrics.items()
                if any(selected_metric in k for selected_metric in metrics)
            }

        fig.add_trace(
            go.Scatterpolar(
                r=list(summary_metrics.values()),
                theta=list(summary_metrics.keys()),
                fill="toself",
                name=title,
            )
        )

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max_score])), showlegend=True
    )

    fig.show()


def plot_bar_plot(eval_results, metrics=None):
    fig = go.Figure()
    data = []

    for eval_result in eval_results:
        title, summary_metrics, _ = eval_result
        if metrics:
            summary_metrics = {
                k: summary_metrics[k]
                for k, v in summary_metrics.items()
                if any(selected_metric in k for selected_metric in metrics)
            }

        data.append(
            go.Bar(
                x=list(summary_metrics.keys()),
                y=list(summary_metrics.values()),
                name=title,
            )
        )

    fig = go.Figure(data=data)

    # Change the bar mode
    fig.update_layout(barmode="group")
    fig.show()

In [3]:
# Model definition
multimodal_model = genai.GenerativeModel( "gemini-1.5-flash",
generation_config= GenerationConfig(temperature=0.5))


In [ ]:
def generate_consultas(num_consultas=150):
    # Ejemplos de consultas para el modelo
    ejemplos_consultas = [
        "Busco una crema para las estrías",
        "Necesito un suplemento de vitamina D para personas mayores",
        "¿Tienes algún producto para la caída del cabello?",
        "Quiero una crema hidratante para piel sensible",
        "¿Hay algún spray nasal para alergias?",
        "¿Tienes algún producto para la caída del cabello?",
        "¿Hay algún producto para el dolor de cabeza?",
        "¿Tienes algún producto para la tos seca?",
        "¿Champú para uso diario que viene en bote rojo?",
        "Enchufe para perro con feromonas para la desesperación"

    ]
    
    # Prompt para el modelo
    prompt = f"""Actúa como un profesional de farmacia y genera consultas VARIADAS en busca de productos de parafarmacia.
    La consulta debe ser similar a estos ejemplos: {ejemplos_consultas}
    Importante: genera consultas diferentes, no te repitas, no uses el mismo ejemplo dos veces.
    Genera solo la consulta, NO agregues viñetas, números ni otros caracteres especiales.
    el output debe contener solo la busqueda."""

    # Generar consultas
    consultas_generadas = []
    for _ in range(num_consultas):
        try:
            response = multimodal_model.generate_content(prompt)
            consulta = response.text.strip()
            consultas_generadas.append(consulta)
        except Exception as e:
            print(f"Error al generar la consulta: {str(e)}")
            consultas_generadas.append("Error en la generación")

    # Crear DataFrame directamente con las consultas
    return pd.DataFrame({'prompt': consultas_generadas})

# Uso de la función
df = generate_consultas() 

In [4]:
# Leemos el archivo CSV existente
df = pd.read_csv('df_evaluation.csv')

# Verificamos que se cargó correctamente
print(f"Número total de filas: {len(df)}")
print("\nPrimeras 5 filas del DataFrame:")
df.head()

Número total de filas: 168

Primeras 5 filas del DataFrame:


,prompt,refined_prompt,bigquery prompt response,bigquery refined prompt response,reranked prompt response,reranked refined prompt response
0,¿Tienen algún producto para la piel seca y con...,"piel seca, picazón, producto","[{'codigo_web': '199572', 'nombre': 'NEUTROGEN...","[{'codigo_web': '199572', 'nombre': 'NEUTROGEN...","[{'codigo_web': '013443', 'nombre': 'Duplo Euc...","[{'codigo_web': '199572', 'nombre': 'NEUTROGEN..."
1,Busco un producto para aliviar la picazón de l...,"Alivio, picazón, picaduras, mosquitos.","[{'codigo_web': '170049', 'nombre': 'AFTER BIT...","[{'codigo_web': '170049', 'nombre': 'AFTER BIT...","[{'codigo_web': '170288', 'nombre': 'AFTER BIT...","[{'codigo_web': '170288', 'nombre': 'AFTER BIT..."
2,Busco un producto para la piel seca que no sea...,"Producto para piel seca, no graso, aroma suave.","[{'codigo_web': '013948', 'nombre': 'Neutrogen...","[{'codigo_web': '013948', 'nombre': 'Neutrogen...","[{'codigo_web': '205921', 'nombre': 'DR. TREE ...","[{'codigo_web': '205921', 'nombre': 'DR. TREE ..."
3,Busco un protector solar para piel sensible y ...,"Protector solar, piel sensible, manchas.","[{'codigo_web': '166570', 'nombre': 'La Roche_...","[{'codigo_web': '154841', 'nombre': 'MELASCREE...","[{'codigo_web': '190308', 'nombre': 'BE+ SKIN ...","[{'codigo_web': '195064', 'nombre': 'BE+ SKIN ..."
4,¿Tienen algún producto para la irritación de l...,"Irritación de la piel, afeitado, productos","[{'codigo_web': '305358', 'nombre': 'Vichy Hom...","[{'codigo_web': '192097', 'nombre': 'MEDICIS S...","[{'codigo_web': '305358', 'nombre': 'Vichy Hom...","[{'codigo_web': '305358', 'nombre': 'Vichy Hom..."


In [ ]:
def refine_query_with_keywords(df):
    # Asegurarse de que existe la columna 'refined_prompt'
    if 'refined_prompt' not in df.columns:
        df['refined_prompt'] = None
    
    # Procesar cada fila del DataFrame
    for index, row in df.iterrows():
        try:
            # Crear el prompt para el modelo usando la consulta de la fila actual
            refinement_prompt = """
            Eres un asistente experto en extracción de palabras clave. 
            
            Consulta del usuario:
            {}
            
            Extrae las palabras clave más importantes de la consulta del usuario y devuélvelas en un formato de texto claro.
            Formato esperado: palabras clave separadas por comas.
            """.format(row['prompt'])

            # Enviar el prompt al modelo para generar el refinamiento
            refinement_response = multimodal_model.generate_content(refinement_prompt)

            # Obtener la respuesta generada y guardarla en la columna refined_prompt
            df.at[index, 'refined_prompt'] = refinement_response.text.strip()
            
            # Imprimir progreso cada 10 filas
            if (index + 1) % 10 == 0:
                print(f"Procesadas {index + 1} consultas...")

        except Exception as e:
            print(f"Error procesando fila {index}: {str(e)}")
            df.at[index, 'refined_prompt'] = "error en el refinamiento"
    
    return df

# Uso de la función
df = refine_query_with_keywords(df)

In [ ]:
# Limpiar el DataFrame eliminando filas con errores en 'refined_prompt'
df = df[~df['refined_prompt'].isin(["error en el refinamiento", "Error en la generación", "Generación, error."])]

# Verificar el número total de filas después de la limpieza
print(f"Número total de filas después de limpiar errores: {len(df)}")
print("\nPrimeras 5 filas del DataFrame limpio:")
print(df.head())

In [56]:
def get_products(prompt):
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_gecko`,  -- Añadidos los backticks
          (SELECT @prompt AS content),
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      d.nombre_matricula_nivel0 AS matricula0,
      d.nombre_matricula_nivel1 AS matricula1,
      d.txt_composicion AS composicion,
      d.forma,
      d.color,
      d.descripcion_visual,
      d.empaque,
      d.zona_de_aplicacion,
      ML.DISTANCE(
        qe.query_embedding,
        d.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_and_embeddings` as d
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """

    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", prompt)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:
        # Asignación de valores con lógica adicional
        descripcion = row.descripcion if row.descripcion else '-'
        modo_implementacion = row.modo_implementacion if row.modo_implementacion else '-'

        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')

        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query,
            "matricula0": row.matricula0,
            "matricula1": row.matricula1,
            "composicion": row.composicion,
            "forma": row.forma,
            "color": row.color,
            "descripcion_visual": row.descripcion_visual,
            "empaque": row.empaque,
            "zona_de_aplicacion": row.zona_de_aplicacion
        })
    return products

In [59]:
def rerank_products(prompt, products):
    ranking_config = discovery_client.ranking_config_path(
        project=PROJECT_ID,
        location=LOCATION,
        ranking_config="default_ranking_config",
    )
    
    records = [
        discoveryengine.RankingRecord(
            id=str(index),
            title=product["nombre"],
            content=" ".join(filter(None, [  # Filtrar valores None y unir con espacios
                str(product["descripcion"] or ""),
                str(product["modo_implementacion"] or ""),
                str(product.get("descripcion_visual", "") or ""),
                str(product.get("matricula0", "") or ""),
                str(product.get("matricula1", "") or ""),
                str(product.get("composicion", "") or ""),
                str(product.get("forma", "") or ""),
                str(product.get("color", "") or ""),
                str(product.get("empaque", "") or ""),
                str(product.get("zona_de_aplicacion", "") or ""),
                str(product.get("codigo_web", "") or ""),
                str(product.get("codigo_nacional", "") or "")
            ]))
        )
        for index, product in enumerate(products)
    ]
    
    request = discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model="semantic-ranker-512@latest",
        top_n=10, # cantidad de productos a rankear
        query=prompt,
        records=records,
    )
    
    response = discovery_client.rank(request=request)
    
    # Aseguramos que los productos están formateados según el esquema
    ranked_products = [
        {
            "codigo_web": products[int(record.id)]["codigo_web"],
            "nombre": products[int(record.id)]["nombre"],
            "codigo_nacional": products[int(record.id)]["codigo_nacional"],
            "descripcion": products[int(record.id)]["descripcion"],
            "modo_implementacion": products[int(record.id)]["modo_implementacion"],
            "imagen_url": products[int(record.id)]["imagen_url"],
            "distance_to_query": products[int(record.id)]["distance_to_query"],
            "descripcion_visual": products[int(record.id)].get("descripcion_visual", ""),
            "matricula0": products[int(record.id)].get("matricula0", ""),
            "matricula1": products[int(record.id)].get("matricula1", ""),
            "composicion": products[int(record.id)].get("composicion", ""),
            "forma": products[int(record.id)].get("forma", ""),
            "color": products[int(record.id)].get("color", ""),
            "empaque": products[int(record.id)].get("empaque", ""),
            "zona_de_aplicacion": products[int(record.id)].get("zona_de_aplicacion", "")
        }
        for record in response.records[:10] # cantidad de productos a mostrar
    ]
    
    return {"products": ranked_products}

In [57]:
# FUNCIONES PARA EVALUAR LAS RESPUESTAS DE BIGQUERY
def execute_bigquery_queries(df):
    # Asegurarse de que existen las nuevas columnas
    if 'bigquery prompt response' not in df.columns:
        df['bigquery prompt response'] = None
    if 'bigquery refined prompt response' not in df.columns:
        df['bigquery refined prompt response'] = None

    # Procesar cada fila del DataFrame
    for index, row in df.iterrows():
        try:
            # Obtener respuesta para el prompt original
            prompt_response = get_products(row['prompt'])
            df.at[index, 'bigquery prompt response'] = prompt_response
            
            # Obtener respuesta para el refined prompt
            refined_response = get_products(row['refined_prompt'])
            df.at[index, 'bigquery refined prompt response'] = refined_response
            
            # Imprimir progreso cada 10 filas
            if (index + 1) % 10 == 0:
                print(f"Procesadas {index + 1} filas...")

        except Exception as e:
            print(f"Error procesando fila {index}: {str(e)}")
            df.at[index, 'bigquery prompt response'] = "error en la consulta"
            df.at[index, 'bigquery refined prompt response'] = "error en la consulta"

    return df

# Uso de la función
df = execute_bigquery_queries(df)

Procesadas 10 filas...
Procesadas 20 filas...
Procesadas 30 filas...
Procesadas 40 filas...
Procesadas 50 filas...
Procesadas 60 filas...
Procesadas 70 filas...
Procesadas 80 filas...
Procesadas 90 filas...
Procesadas 100 filas...
Procesadas 120 filas...
Procesadas 130 filas...
Procesadas 140 filas...
Procesadas 150 filas...
Procesadas 160 filas...
Procesadas 170 filas...


In [60]:
def execute_reranking(df):
    # Asegurarse de que existen las nuevas columnas
    if 'reranked prompt response' not in df.columns:
        df['reranked prompt response'] = None
    if 'reranked refined prompt response' not in df.columns:
        df['reranked refined prompt response'] = None

    print("Iniciando el proceso de reranking...", flush=True)

    # Procesar cada fila del DataFrame
    for index, row in df.iterrows():
        try:
            # Reranking usando el prompt original con sus productos correspondientes
            if isinstance(row['bigquery prompt response'], list):  # Verificar que hay productos para rankear
                reranked_original = rerank_products(
                    prompt=row['prompt'],
                    products=row['bigquery prompt response']
                )
                df.at[index, 'reranked prompt response'] = reranked_original['products']
            
            # Reranking usando el refined_prompt con sus productos correspondientes
            if isinstance(row['bigquery refined prompt response'], list):  # Verificar que hay productos para rankear
                reranked_refined = rerank_products(
                    prompt=row['refined_prompt'],
                    products=row['bigquery refined prompt response']
                )
                df.at[index, 'reranked refined prompt response'] = reranked_refined['products']
            
            # Imprimir progreso cada 10 filas
            if (index + 1) % 10 == 0:
                print(f"Procesadas {index + 1} filas...", flush=True)

        except Exception as e:
            print(f"Error procesando fila {index}: {str(e)}", flush=True)
            df.at[index, 'reranked prompt response'] = "error en el reranking"
            df.at[index, 'reranked refined prompt response'] = "error en el reranking"

    return df

# Uso de la función
df = execute_reranking(df)

Iniciando el proceso de reranking...


Procesadas 10 filas...
Procesadas 20 filas...
Procesadas 30 filas...
Procesadas 40 filas...
Procesadas 50 filas...
Procesadas 60 filas...
Procesadas 70 filas...
Procesadas 80 filas...
Procesadas 90 filas...
Procesadas 100 filas...
Procesadas 120 filas...
Procesadas 130 filas...
Procesadas 140 filas...
Procesadas 150 filas...
Procesadas 160 filas...
Procesadas 170 filas...


In [12]:

def rerank_products_gemini(prompt, products):
    try:
        rerank_prompt = f"""
        Eres un clasificador de productos farmacéuticos.
        
        CONSULTA DEL USUARIO: "{prompt}"
        
        PRODUCTOS A CLASIFICAR: {json.dumps(products, ensure_ascii=False)}
        
        INSTRUCCIONES:
        1. Analiza los productos proporcionados
        2. Selecciona y ordena los productos más relevantes para la consulta
        3. Devuelve SOLO una lista JSON con los productos seleccionados
        4. Mantén exactamente la misma estructura de datos para cada producto
        5. No modifiques ninguna información de los productos
        
        IMPORTANTE: Tu respuesta debe ser ÚNICAMENTE una lista JSON de productos, sin texto adicional.
        """

        print("\nPrompt enviado a Gemini:")
        print(rerank_prompt)

        response = multimodal_model.generate_content(rerank_prompt)
        
        print("\nRespuesta recibida de Gemini:")
        print(response.text)

        if not response or not response.text:
            raise ValueError("Respuesta vacía del modelo Gemini")
            
        try:
            ranked_products = json.loads(response.text)
            return {"products": ranked_products}
            
        except json.JSONDecodeError as e:
            print(f"\nError al parsear JSON. Respuesta recibida:\n{response.text}")
            raise ValueError(f"Error al parsear JSON: {str(e)}")
            
    except Exception as e:
        print(f"\nError en rerank_products_gemini: {str(e)}")
        return {"error": f"Error al rerankear los productos: {str(e)}"}

In [ ]:
import ast
import re

def execute_reranking_gemini(df):
    if 'reranked prompt gemini response' not in df.columns:
        df['reranked prompt gemini response'] = None
    if 'reranked refined prompt gemini response' not in df.columns:
        df['reranked refined prompt gemini response'] = None

    print("Iniciando el proceso de reranking con Gemini...", flush=True)

    for index, row in df.iterrows():
        try:
            # Procesar prompt original
            if row['bigquery prompt response']:
                # Usar ast.literal_eval para una conversión más segura
                try:
                    products = ast.literal_eval(row['bigquery prompt response'])
                except:
                    # Si ast.literal_eval falla, intentar limpiar el string
                    clean_str = row['bigquery prompt response']
                    clean_str = clean_str.replace('\n', '\\n')
                    clean_str = clean_str.replace('\r', '\\r')
                    clean_str = clean_str.replace('\t', '\\t')
                    # Escapar las comillas dobles dentro del texto
                    clean_str = re.sub(r'(?<!\\)"', '\\"', clean_str)
                    # Reemplazar comillas simples por dobles
                    clean_str = clean_str.replace("'", '"')
                    products = json.loads(clean_str)

                reranked_original = rerank_products_gemini(
                    prompt=row['prompt'],
                    products=products
                )
                df.at[index, 'reranked prompt gemini response'] = reranked_original['products']
            
            # Procesar refined prompt
            if row['bigquery refined prompt response']:
                # Usar el mismo proceso de conversión segura
                try:
                    refined_products = ast.literal_eval(row['bigquery refined prompt response'])
                except:
                    clean_str = row['bigquery refined prompt response']
                    clean_str = clean_str.replace('\n', '\\n')
                    clean_str = clean_str.replace('\r', '\\r')
                    clean_str = clean_str.replace('\t', '\\t')
                    clean_str = re.sub(r'(?<!\\)"', '\\"', clean_str)
                    clean_str = clean_str.replace("'", '"')
                    refined_products = json.loads(clean_str)

                reranked_refined = rerank_products_gemini(
                    prompt=row['refined_prompt'],
                    products=refined_products
                )
                df.at[index, 'reranked refined prompt gemini response'] = reranked_refined['products']
            
            if (index + 1) % 10 == 0:
                print(f"Procesadas {index + 1} filas...", flush=True)

        except Exception as e:
            print(f"Error procesando fila {index}: {str(e)}", flush=True)
            df.at[index, 'reranked prompt gemini response'] = "error en el reranking"
            df.at[index, 'reranked refined prompt gemini response'] = "error en el reranking"

    return df
df = execute_reranking_gemini(df)

In [ ]:
# ----------------------Evaluation Dataset----------------------

"""To evaluate the RAG generated answers,
the evaluation dataset is required to contain the following fields:

Prompt: The user supplied prompt consisting of the User Question and the RAG Retrieved Context
Response: The RAG Generated Answer

Your dataset must include a minimum of one evaluation example.
We recommend around 100 examples to ensure high-quality aggregated metrics
and statistically significant results."""

def create_evaluation_scenarios(df):
    # Escenario 1: BigQuery con prompt original
    eval_bigquery_prompt = pd.DataFrame({
        "prompt": ["Answer the question: " + row['prompt'] + " Context: " + row['bigquery prompt response'] 
                  for _, row in df.iterrows()],
        "response": df['bigquery prompt response'].tolist()
    })

    # Escenario 2: BigQuery con refined prompt
    eval_bigquery_refined = pd.DataFrame({
        "prompt": ["Answer the question: " + row['refined_prompt'] + " Context: " + row['bigquery refined prompt response'] 
                  for _, row in df.iterrows()],
        "response": df['bigquery refined prompt response'].tolist()
    })

    # Escenario 3: Reranker con respuestas de BigQuery prompt
    eval_rerank_prompt = pd.DataFrame({
        "prompt": ["Answer the question: " + row['prompt'] + " Context: " + row['reranked prompt response'] 
                  for _, row in df.iterrows()],
        "response": df['reranked prompt response'].tolist()
    })

    # Escenario 4: Reranker con respuestas de BigQuery refined prompt
    eval_rerank_refined = pd.DataFrame({
        "prompt": ["Answer the question: " + row['refined_prompt'] + " Context: " + row['reranked refined prompt response'] 
                  for _, row in df.iterrows()],
        "response": df['reranked refined prompt response'].tolist()
    })

    # Escenario 5: Gemini con respuestas del Reranker prompt
    eval_gemini_prompt = pd.DataFrame({
        "prompt": ["Answer the question: " + row['prompt'] + " Context: " + row['reranked prompt gemini response'] 
                  for _, row in df.iterrows()],
        "response": df['reranked prompt gemini response'].tolist()
    })

    # Escenario 6: Gemini con respuestas del Reranker refined prompt
    eval_gemini_refined = pd.DataFrame({
        "prompt": ["Answer the question: " + row['refined_prompt'] + " Context: " + row['reranked refined prompt gemini response'] 
                  for _, row in df.iterrows()],
        "response": df['reranked refined prompt gemini response'].tolist()
    })

    return {
        'bigquery_prompt': eval_bigquery_prompt,
        'bigquery_refined': eval_bigquery_refined,
        'rerank_prompt': eval_rerank_prompt,
        'rerank_refined': eval_rerank_refined,
        'gemini_prompt': eval_gemini_prompt,
        'gemini_refined': eval_gemini_refined
    }

In [ ]:
# ----------------------Evaluation Dataset SIN GEMINI----------------------

"""To evaluate the RAG generated answers,
the evaluation dataset is required to contain the following fields:

Prompt: The user supplied prompt consisting of the User Question and the RAG Retrieved Context
Response: The RAG Generated Answer

Your dataset must include a minimum of one evaluation example.
We recommend around 100 examples to ensure high-quality aggregated metrics
and statistically significant results."""

def create_evaluation_scenarios(df):
    # Escenario 1: BigQuery con prompt original
    eval_bigquery_prompt = pd.DataFrame({
        "prompt": ["Answer the question: " + row['prompt'] + " Context: " + row['bigquery prompt response'] 
                  for _, row in df.iterrows()],
        "response": df['bigquery prompt response'].tolist()
    })

    # Escenario 2: BigQuery con refined prompt
    eval_bigquery_refined = pd.DataFrame({
        "prompt": ["Answer the question: " + row['refined_prompt'] + " Context: " + row['bigquery refined prompt response'] 
                  for _, row in df.iterrows()],
        "response": df['bigquery refined prompt response'].tolist()
    })

    # Escenario 3: Reranker con respuestas de BigQuery prompt
    eval_rerank_prompt = pd.DataFrame({
        "prompt": ["Answer the question: " + row['prompt'] + " Context: " + row['reranked prompt response'] 
                  for _, row in df.iterrows()],
        "response": df['reranked prompt response'].tolist()
    })

    # Escenario 4: Reranker con respuestas de BigQuery refined prompt
    eval_rerank_refined = pd.DataFrame({
        "prompt": ["Answer the question: " + row['refined_prompt'] + " Context: " + row['reranked refined prompt response'] 
                  for _, row in df.iterrows()],
        "response": df['reranked refined prompt response'].tolist()
    })


    return {
        'bigquery_prompt': eval_bigquery_prompt,
        'bigquery_refined': eval_bigquery_refined,
        'rerank_prompt': eval_rerank_prompt,
        'rerank_refined': eval_rerank_refined,
    }


1. question_answering_quality
¿Las respuestas son relevantes para la pregunta del usuario?
¿Se proporcionan los detalles importantes del producto?
¿La información es precisa y útil?

2. groundedness
¿Las respuestas se basan en la información proporcionada en el contexto?
¿Se evitan afirmaciones no respaldadas por los datos?
¿Las recomendaciones de productos son fieles a sus descripciones?

3. coherence
¿Las respuestas están bien estructuradas?
¿La información fluye de manera lógica?
¿Las recomendaciones tienen sentido en el contexto de la pregunta?

4. instruction_following
¿Las respuestas siguen el formato esperado?
¿Se mantiene el enfoque en la consulta del usuario?
¿Se respetan las restricciones del dominio farmacéutico?

También podríamos incluir evaluaciones comparativas usando:

5. pairwise_question_answering_quality
Comparar la calidad entre respuestas de prompt original vs refined
Evaluar la efectividad del reranking vs las respuestas directas de BigQuery
Medir el valor añadido por Gemini en la generación de respuestas

In [ ]:
def prepare_pairwise_evaluation_dataframes(evaluation_scenarios):
    # 1. Comparación de BigQuery: prompt vs refined_prompt
    eval_bigquery_comparison = evaluation_scenarios['bigquery_refined'].copy()
    eval_bigquery_comparison['baseline_model_response'] = evaluation_scenarios['bigquery_prompt']['response']

    # 2. Comparación de Reranker: prompt vs refined_prompt
    eval_rerank_comparison = evaluation_scenarios['rerank_refined'].copy()
    eval_rerank_comparison['baseline_model_response'] = evaluation_scenarios['rerank_prompt']['response']

    # 3. Comparación BigQuery vs Reranker
    # 3.1 Para prompt original
    eval_bigquery_vs_rerank_prompt = evaluation_scenarios['rerank_prompt'].copy()
    eval_bigquery_vs_rerank_prompt['baseline_model_response'] = evaluation_scenarios['bigquery_prompt']['response']
    
    # 3.2 Para refined prompt
    eval_bigquery_vs_rerank_refined = evaluation_scenarios['rerank_refined'].copy()
    eval_bigquery_vs_rerank_refined['baseline_model_response'] = evaluation_scenarios['bigquery_refined']['response']

    # 4. Comparación de Gemini: prompt vs refined_prompt
    eval_gemini_comparison = evaluation_scenarios['gemini_refined'].copy()
    eval_gemini_comparison['baseline_model_response'] = evaluation_scenarios['gemini_prompt']['response']

    # 5. Comparación Gemini vs otros modelos
    # 5.1 Gemini vs Reranker (prompt)
    eval_gemini_vs_rerank_prompt = evaluation_scenarios['gemini_prompt'].copy()
    eval_gemini_vs_rerank_prompt['baseline_model_response'] = evaluation_scenarios['rerank_prompt']['response']
    
    # 5.2 Gemini vs Reranker (refined)
    eval_gemini_vs_rerank_refined = evaluation_scenarios['gemini_refined'].copy()
    eval_gemini_vs_rerank_refined['baseline_model_response'] = evaluation_scenarios['rerank_refined']['response']
    
    # 5.3 Gemini vs BigQuery (prompt)
    eval_gemini_vs_bigquery_prompt = evaluation_scenarios['gemini_prompt'].copy()
    eval_gemini_vs_bigquery_prompt['baseline_model_response'] = evaluation_scenarios['bigquery_prompt']['response']
    
    # 5.4 Gemini vs BigQuery (refined)
    eval_gemini_vs_bigquery_refined = evaluation_scenarios['gemini_refined'].copy()
    eval_gemini_vs_bigquery_refined['baseline_model_response'] = evaluation_scenarios['bigquery_refined']['response']

    return {
        'bigquery_comparison': eval_bigquery_comparison,
        'rerank_comparison': eval_rerank_comparison,
        'bigquery_vs_rerank_prompt': eval_bigquery_vs_rerank_prompt,
        'bigquery_vs_rerank_refined': eval_bigquery_vs_rerank_refined,
        'gemini_comparison': eval_gemini_comparison,
        'gemini_vs_rerank_prompt': eval_gemini_vs_rerank_prompt,
        'gemini_vs_rerank_refined': eval_gemini_vs_rerank_refined,
        'gemini_vs_bigquery_prompt': eval_gemini_vs_bigquery_prompt,
        'gemini_vs_bigquery_refined': eval_gemini_vs_bigquery_refined
    }

In [ ]:
# ----------------------Preparar Pairwise Evaluation sin GEMINI----------------------

def prepare_pairwise_evaluation_dataframes(evaluation_scenarios):
    # 1. Comparación de BigQuery: prompt vs refined_prompt
    eval_bigquery_comparison = evaluation_scenarios['bigquery_refined'].copy()
    eval_bigquery_comparison['baseline_model_response'] = evaluation_scenarios['bigquery_prompt']['response']

    # 2. Comparación de Reranker: prompt vs refined_prompt
    eval_rerank_comparison = evaluation_scenarios['rerank_refined'].copy()
    eval_rerank_comparison['baseline_model_response'] = evaluation_scenarios['rerank_prompt']['response']

    # 3. Comparación BigQuery vs Reranker
    # 3.1 Para prompt original
    eval_bigquery_vs_rerank_prompt = evaluation_scenarios['rerank_prompt'].copy()
    eval_bigquery_vs_rerank_prompt['baseline_model_response'] = evaluation_scenarios['bigquery_prompt']['response']
    
    # 3.2 Para refined prompt
    eval_bigquery_vs_rerank_refined = evaluation_scenarios['rerank_refined'].copy()
    eval_bigquery_vs_rerank_refined['baseline_model_response'] = evaluation_scenarios['bigquery_refined']['response']

    return {
        'bigquery_comparison': eval_bigquery_comparison,
        'rerank_comparison': eval_rerank_comparison,
        'bigquery_vs_rerank_prompt': eval_bigquery_vs_rerank_prompt,
        'bigquery_vs_rerank_refined': eval_bigquery_vs_rerank_refined,

    }